# A Descriptive Analysis of 2025 Flight Activity, Flight Traffic and Airport Network Patterns

### Research report, data preparation and interactive analysis

**Dataset:** EUROCONTROL dataset: https://www.opdi.aero/flight-list-data

**Study period:** This study is targeted for the 2025 year, due to data size limitations and incomplete data for 2026. Hence we target the closest completed year for which the data is complete.  

**Companion application:** As a byproduct of this study, a streamlit application is built to give more insight into the data and build a dashboard that allows for data analysis and decision making.

---

# Abstract

This project investigates patterns in observed flight activity during 2025 using the Open Performance Data Initiative (OPDI) Flight List dataset. It focuses on how recorded flights are distributed over time, across operators and aircraft types, and between departure and destination airports. A further aim is to construct a descriptive picture of the airport network represented in the data.

The project proposes that a large-scale, carefully prepared flight-list dataset can provide a useful descriptive view of air traffic activity, provided that the limitations of surveillance-derived records and missing airport information are made explicit. Twelve monthly Parquet files are combined and standardised, repeated records are checked, airport codes are matched to a reference dataset, and records are aggregated into monthly, airport, operator, aircraft and directional-route summaries. The resulting datasets support both the analysis in this notebook and an interactive Streamlit dashboard.

The previous full-year processing run retained 16,264,265 unique flight-list records after removing 2,828 exact duplicates. However, only 5,578,760 records (34.3%) contained both a departure airport (ADEP) and destination airport (ADES). This has a direct effect on the route analysis: route results describe the endpoint-complete subset rather than the full set of accepted records. Of 86,928 distinct directional airport pairs, 78,462 (90.3%) had coordinates for both airports and could be represented on the network map.

The project is descriptive rather than predictive or causal. The available data supports analysis of recorded activity, but it does not provide the scheduled timestamps required to measure airline punctuality or delay. Similarly, the interval between `first_seen` and `last_seen` is treated as an observation span, not as confirmed airborne or gate-to-gate time.

**Keywords:** air traffic, flight data, airport activity, route network, ADS-B, OPDI, OpenSky Network, descriptive analysis


# 1. Introduction

## 1.1 Background

Air traffic is a complex transport system in which airlines, aircraft, airports and routes interact across time and geography. Understanding how activity is distributed across these elements is relevant to several areas of aviation analysis, including airport planning, airline network analysis, fleet deployment and the study of changes in traffic patterns.

Large-scale flight datasets make it possible to investigate these patterns using observed flight records rather than relying only on aggregated annual statistics. In particular, surveillance-derived datasets can provide information about aircraft, operators, departure and destination airports, and the timing of observed trajectories. When organised consistently, these records can be used to describe the structure and distribution of air traffic over a defined period.

However, the availability of a large number of records does not automatically make the data complete or suitable for every aviation question. Individual fields may be missing, airport codes may not always be identifiable, and observation timestamps do not necessarily correspond to operational milestones such as take-off, landing, off-block or in-block. A sound analysis must therefore begin by understanding the source and defining exactly what each indicator represents.

## 1.2 Motivation

The motivation for this project is to build a practical and reproducible way to explore flight activity at more than one level. Rather than focusing on a single airline or airport, the analysis considers the wider dataset and examines how activity can be viewed through several connected perspectives:

- **Temporal perspective:** how recorded activity changes over the months of 2025.
- **Operator perspective:** how flight-list records are distributed across operators.
- **Airport perspective:** which airports appear most frequently as departure and destination points.
- **Fleet perspective:** which aircraft type designators occur most often.
- **Network perspective:** which directional airport pairs appear in the records and how these connections are distributed geographically.

These perspectives are all commonly related and they provide a complete picture of the flight network within the EU sector. For example, an airport's activity is partly shaped by the routes connected to it, while route activity may be associated with particular operators and aircraft types. Bringing these dimensions together creates a more complete descriptive picture than examining each one in isolation.\]
'
m ## 1.3 Project proposal

**This  project proposes to develop a reproducible descriptive analysis of European flight activity during 2025 using the OPDI Flight List dataset, and to translate the results into an interactive dashboard for exploration.**

The proposal is based on three ideas. First, the monthly source files should be combined and prepared through a transparent process so that the resulting counts are consistent. Second, the analysis should distinguish between the full set of accepted flight-list records and the smaller subsets that can be used for particular questions, especially route analysis. Third, the findings should be presented in a form that allows users to explore activity by time, operator, airport, aircraft type and route.

The intended outcome is not a model that predicts future traffic or evaluates airline performance. It is a structured description of the activity represented in the dataset, supported by clear data definitions, quality checks and an interactive means of exploring the results.

## 1.4 Aim and objectives

The overall aim is:

**To describe the distribution and structure of observed flight-list activity during 2025, with particular attention to temporal patterns, operators, airports, aircraft types and directional airport connections.**

The project has the following objectives:

1. Combine the twelve monthly OPDI Flight List files into a consistent 2025 dataset.
2. Assess the completeness and reliability of the main fields used in the analysis.
3. Prepare consistent summaries of monthly, operator, airport and aircraft-type activity.
4. Identify directional airport pairs and examine the airport network represented by those pairs.
5. Evaluate how missing or unmatched airport information affects the coverage of route and map-based results.
6. Develop a Streamlit dashboard that allows the prepared data and findings to be explored interactively.

## 1.5 Research questions

The analysis is guided by the following research questions:

**[RQ1] Temporal activity:** How is observed flight-list activity distributed across the months of 2025?

**[RQ2] Operators and fleet:** How are the accepted records distributed across operators and aircraft types, and which operators and aircraft types account for the largest shares of the records with those fields available?

**[RQ3] Airport activity:** Which airports appear most frequently as departure and destination airports in the dataset, and how do departure and arrival counts compare?

**[RQ4] Route network:** Which directional airport-to-airport connections occur most frequently, and what does the resulting airport network look like geographically?

**[RQ5] Data completeness:** To what extent do missing or unmatched fields, particularly ADEP and ADES, limit the coverage and interpretation of the results?

These questions are deliberately descriptive. They ask what patterns are visible in the available records, rather than why those patterns occur or what would happen in a future period.

## 1.6 Scope and boundaries

The study covers the calendar year 2025, using the twelve monthly OPDI Flight List files. Its geographical coverage is determined by the flights represented in those files and should not be assumed to be a complete census of all European or global flights.

The analysis concerns records in the OPDI Flight List. A record is treated as an identified flight-list trajectory record, not automatically as an independently verified airport movement, passenger journey or complete operational flight record.

Unfortunately, due to the way the data is structured, the project does not attempt to:
- estimate passenger numbers, load factors or seat capacity;
- calculate scheduled delay or on time performance;
- reconstruct actual aircraft trajectories from airport-to-airport map connections;
- infer the reasons behind differences between operators or airports;
- forecast future traffic.

These data would require ADS-B data, which is typically findibale under open-source repositories or paid services. OpenSky is the main alternative source to consider, however an application to access their data temporarily for this study was denied. 

Link to this alternative dataset: https://opensky-network.org/data



# 2. Data source and conceptual understanding

## 2.1 Open Performance Data Initiative

The project uses the **Open Performance Data Initiative (OPDI)**, developed through EUROCONTROL's Performance Review Commission in collaboration with the OpenSky Network.

OPDI provides related flight, event and measurement data. The Flight List is used here because it provides a high-level record for an identified flight trajectory, with fields that can describe the aircraft, operator, airport endpoints and observation times.

The data is derived from surveillance information collected through the OpenSky Network. It is therefore useful for examining observed flight activity at scale, but it should not automatically be interpreted as a complete official record of all flights. Surveillance coverage and the availability of particular fields can vary.

**Primary documentation**

- OPDI data catalogue: https://www.opdi.aero/data
- Flight List downloads: https://www.opdi.aero/flight-list-data
- OPDI concepts: https://www.opdi.aero/concepts
- OPDI methodology: https://www.opdi.aero/methodology
- OPDI research paper, SESAR Innovation Days 2024: https://sesar.eu/sites/default/files/documents/sid/2024/papers/SIDs_2024_paper_100%20final.pdf

## 2.2 Main variables

| Variable | Meaning | Role in this study |
|---|---|---|
| `id` | Identifier assigned to an OPDI flight record | 
| `dof` / `flight_date` | Date associated with the flight |
| `icao_operator` | ICAO operator designator, if available | 
| `icao24` | 24-bit aircraft transponder address | 
| `adep` | Aerodrome of departure |
| `ades` | Aerodrome of destination | 
| `typecode` | Aircraft type designator | 
| `model` | Aircraft model description | 
| `registration` | Aircraft registration | 
| `first_seen` | First observation time associated with the trajectory | 
| `last_seen` | Last observation time associated with the trajectory | 

**ADEP** and **ADES** are important throughout the project. ADEP identifies the departure aerodrome recorded by OPDI, while ADES identifies the destination aerodrome. **ICAO** is the International Civil Aviation Organization. ICAO airport location indicators generally contain four letters; IATA airport codes, which are familiar from passenger tickets and airport displays, generally contain three.

The `icao24` field is a 24-bit address broadcast by an aircraft transponder. It is useful for distinguishing aircraft addresses in the dataset, but should not be confused with an aircraft registration or a guaranteed permanent identifier for an individual airframe.

The `typecode` field identifies an aircraft type using a standard designator. It allows the analysis to describe the aircraft types represented in the records, although missing or non-standard values need to be treated carefully.

## 2.3 Interpreting observation timestamps

The project calculates an observation span:

`observation_span_min = last_seen - first_seen`

This is the elapsed time between the first and last observations associated with a flight-list record. OPDI's concepts documentation describes the first observation as associated with the departure aerodrome, usually around off-block, and the last observation as associated with the destination aerodrome before in-block.

Nevertheless, the difference between these timestamps should not be treated as a precise measure of airborne time or gate-to-gate time. The timestamps do not necessarily represent take-off and landing, and they do not provide all operational milestones required to calculate block time. The analysis therefore uses the term **observation span** consistently.

## 2.4 Supporting airport reference data

A separate airport reference file is used to add airport names and geographic coordinates to ADEP and ADES codes. This makes it possible to create airport summaries and plot airport-pair connections.

The reference file is a supporting source, not a replacement for the original flight data. When an airport code cannot be matched unambiguously, the flight record is retained for analyses that do not require coordinates. The code is excluded only from map-based calculations that require a reliable location.


# 3. Research methodology

## 3.1 Research approach

The project follows a quantitative, descriptive approach. The observations are grouped and counted to identify patterns across time, operators, airports, aircraft types and routes. No causal relationship is estimated, and no prediction is made.

The main unit of analysis is an **OPDI flight-list record**. This is the record counted in the overall totals and in the relevant groupings. The unit changes only when a question explicitly requires a different entity: for example, route analysis counts records grouped by an ordered airport pair, while aircraft diversity counts distinct `icao24` values or typecodes.

This distinction is important because a flight-list record, an airport movement, an aircraft and a route are not interchangeable units.

## 3.2 Operational definitions and indicators

| Research dimension | Indicator | Definition |
|---|---|---|
| Overall activity | Accepted records | Number of unique accepted flight-list IDs |
| Temporal activity | Monthly records | Number of accepted records assigned to each month |
| Operator activity | Records per operator | Number of records with a given operator code |
| Departure activity | Departures per airport | Number of records with a given ADEP |
| Arrival activity | Arrivals per airport | Number of records with a given ADES |
| Airport activity | Departure plus arrival records | Sum of the two endpoint counts |
| Route activity | Records per directional pair | Number of records grouped by ordered ADEP and ADES |
| Network coverage | Mapped directional pairs | Route pairs for which both airport coordinates are available |
| Fleet profile | Records per aircraft type | Number of records with a given typecode |
| Observation timing | Observation span | Difference between `last_seen` and `first_seen` |

A percentage requires a clear **denominator**: the population against which the percentage is calculated. For example, an operator's share is calculated among records with a known operator code. A route's share is calculated among records with both airport endpoints. These percentages should not be presented as shares of all accepted records unless the denominator is in fact the full accepted dataset.

## 3.3 Directional route definition

A route is represented as an ordered pair:

**ADEP - ADES**

The direction is retained because Amsterdam to London and London to Amsterdam are different directional services. Combining the two would remove information about the direction of travel and could hide differences in traffic between the two directions.

A route record is included only when both ADEP and ADES are present. Records missing either endpoint remain in the overall dataset and can still contribute to other analyses.

## 3.4 Data preparation and quality decisions

The monthly files are checked, read and combined. Column names and text codes are standardised, and date and timestamp fields are converted to consistent datetime values. Exact duplicate rows are removed before any summaries are calculated.

If the same flight ID remains more than once with different information, it is treated as a conflicting ID. Rather than arbitrarily selecting one version, the ID is excluded from the accepted dataset and written to an exception file. This avoids silently deciding which conflicting record is correct.

Airport codes are matched against the reference file. ICAO codes are used where possible, with IATA codes used as a fallback where the source supports it. If a code maps to multiple different airport locations, it is marked ambiguous rather than assigned coordinates by guesswork.

Importantly, missing fields do not lead to automatic deletion of the whole record. A record with a missing destination can still contribute to the overall record count, monthly totals, or an operator analysis if those fields are present. It is excluded only from calculations that require the missing information.

## 3.5 Why these preparation choices matter

The choices above protect the interpretation of the results:

- Removing exact duplicates prevents the same row from inflating counts.
- Excluding conflicting IDs avoids arbitrary decisions about which version is correct.
- Keeping incomplete records preserves useful information for analyses that do not depend on the missing fields.
- Treating ambiguous airport matches cautiously avoids creating misleading geographic connections.
- Keeping route direction preserves the distinction between traffic in opposite directions.
- Defining the denominator for each result prevents percentages from being interpreted as shares of a larger population than the data supports.

## 3.6 Validation

The prepared data is checked for internal consistency. The principal checks are:

1. Each accepted flight ID appears once.
2. Each directional airport pair appears once in the route summary.
3. The sum of route counts equals the number of accepted records with both ADEP and ADES.
4. Mapped airport coordinates fall within valid latitude and longitude ranges.
5. Airport departure and arrival summaries reconcile with the number of records having known ADEP and ADES values.

These checks indicate whether the preparation and aggregation have been performed consistently. They do not prove that the source dataset captures every flight or that every source field is correct.


# 4. Analysis framework

The analysis is organised around the five research questions. Each question uses the same accepted flight-list dataset, but the relevant subset changes according to the fields required.

## 4.1 Temporal analysis  [RQ1]

The records are grouped by month to describe how observed activity is distributed throughout 2025. Monthly counts provide a simple overview of the temporal profile. Where useful, the analysis also compares the number of distinct operators, aircraft addresses and routes represented in each month.

These are counts of records observed in the dataset. They should not be interpreted as official airport movement statistics or as proof of a seasonal cause.

## 4.2 Operator and aircraft analysis  [RQ2]

Operator activity is described by counting records with a known `icao_operator` value. Aircraft activity is described using `typecode`, with `icao24` available as an aircraft-address measure.

The analysis reports the most frequent categories and their shares among records where the relevant field is known. Missing operator or aircraft information is reported separately so that the rankings are not mistaken for complete coverage.

## 4.3 Airport analysis  [RQ3]

Departure and arrival records are grouped separately by airport. This distinction is retained because an airport may have different levels of recorded departure and arrival activity.

The combined airport movement indicator is calculated as departures plus arrivals. It is a convenient summary of the two endpoint counts in this dataset, but it is not a separately verified count of official airport movements.

## 4.4 Route-network analysis  [RQ4]

Records with both ADEP and ADES are grouped into directional airport pairs. The resulting table describes how frequently each airport-to-airport connection occurs in the data.

For the map, each airport is represented by its reference coordinates and each route is represented by a line connecting the two airport locations. These lines visualise the airport-pair network; they are not reconstructed aircraft trajectories, airways or actual flown paths.

## 4.5 Data completeness analysis [RQ5]

The completeness analysis measures how often the fields required for each question are available. Particular attention is given to ADEP and ADES because both are necessary to identify a route.

The analysis distinguishes three different quantities:

- **Record count:** number of flight-list records.
- **Code count:** number of distinct airport codes.
- **Route pair count:** number of distinct ordered ADEP–ADES combinations.



# 5. Results

## 5.1 Baseline processing results

The figures in this section are taken from the previous full-year processing run, recorded in `preprocessing_report.json` on 5 November 2026. They are included as baseline results for comparison with the outputs generated when the revised notebook is run. They are not claimed to be the result of a fresh execution of this version.

| Measure | Baseline result |
|---|---:|
| Monthly input files | 12 |
| Rows entering duplicate handling | 16,267,093 |
| Exact duplicate rows removed | 2,828 |
| Conflicting flight IDs excluded | 0 |
| Accepted unique flight-list records | **16,264,265** |
| Accepted records with both ADEP and ADES | 5,578,760 (34.3%) |
| Accepted records missing at least one route endpoint | 10,685,505 (65.7%) |
| Distinct directional airport pairs | 86,928 |
| Directional pairs with both airports mapped | 78,462 (90.3%) |
| Unmapped directional pairs | 8,466 (9.7%) |
| Distinct airport codes | 1,115 |
| Airport codes unmatched or ambiguous | 190 |
| Accepted records without a usable flight date | 0 |

## 5.2 Record volume and duplicate handling

The twelve monthly files contained 16,267,093 rows entering duplicate handling. Removing 2,828 exact duplicate rows resulted in 16,264,265 accepted unique records. No conflicting IDs were found in the baseline run.

The number of duplicates is small relative to the full dataset, but removing them remains important. Even a small number of repeated records can affect specific route or operator counts if those duplicates are concentrated in a particular group.

## 5.3 Completeness of airport endpoints

A total of 5,578,760 accepted records contained both ADEP and ADES, equivalent to 34.3% of all accepted records. The remaining 10,685,505 records (65.7%) were missing at least one endpoint.

This is a central finding because route analysis requires both airport codes. The route analysis therefore covers a subset of the full dataset. It would be misleading to describe the route rankings as a complete representation of all 16.26 million accepted records.

## 5.4 Airport matching and network coverage

The records with both airport endpoints produced 86,928 distinct directional airport pairs. Of these, 78,462 (90.3%) had coordinates for both airports and could be included in the mapped network. The other 8,466 pairs (9.7%) could not be fully mapped.

There were 1,115 distinct airport codes in the flight data, of which 190 were unmatched or ambiguous. This is a count of different codes, not a count of flights. It would therefore be incorrect to interpret 190 out of 1,115 as the percentage of flight records that could not be mapped. Airport codes can occur with very different frequencies.

## 5.5 Initial interpretation

The baseline results show that the dataset is large enough to support broad descriptive analysis, but field completeness differs substantially across variables. In particular, route analysis is limited by the availability of both airport endpoints.

This means the data-quality analysis is not just a technical check. It directly affects the research conclusions. Any route or airport-network result should be accompanied by a clear statement of the number of records and route pairs on which it is based.

The baseline figures alone do not explain why airport endpoints are missing. A further investigation could examine whether missingness is concentrated by month, operator, aircraft type or another field. That would help determine whether the limitation is evenly distributed or more pronounced in particular parts of the dataset.


# 6. Discussion

The project demonstrates how a large surveillance-derived flight-list dataset can be transformed into a structured descriptive view of air traffic. The analysis links temporal activity, operator records, aircraft types, airport counts and directional airport connections within one consistent data-preparation process.

The most important methodological finding is the difference between the full accepted dataset and the subset suitable for route analysis. Although more than sixteen million records are retained, only 34.3% have both airport endpoints. This does not make the remaining records useless: they can still contribute to other analyses where the necessary fields are present. It does, however, mean that the route network cannot be treated as a complete representation of all accepted records.

The airport matching results introduce a second distinction. A route pair can be present in the flight data but still be unavailable for mapping if one or both airport codes cannot be matched reliably to coordinates. The 90.3% mapping figure refers to distinct directional route pairs, not to flight records. A small number of frequently used unmatched routes could represent many records, while a large number of rare unmatched routes could represent relatively few. A record-weighted mapping coverage measure would therefore be a useful additional indicator.

The dashboard makes these distinctions easier to explore. Users can move between the overall dataset and selected months, operators, airports, aircraft types or routes. However, the dashboard should be treated as an interface to the analysis, not as a separate source of evidence. Its figures depend on the prepared datasets and the definitions established in this notebook.

Overall, the results support a descriptive analysis of observed flight-list activity. They do not support conclusions about airline efficiency, punctuality, passenger demand or the causes of traffic patterns without additional data and a different research design.


# 7. Limitations and further research

## 7.1 Data coverage

The source is derived from surveillance data. It should not be assumed to be a complete census of all flights. Coverage may vary across locations, periods and aircraft, and missing fields may affect some analyses more than others.

## 7.2 Missing airport endpoints

Only 34.3% of accepted records in the baseline run contained both ADEP and ADES. This restricts the population used for route and network analysis. Further work should investigate whether missing endpoints are associated with particular operators, months, aircraft types or geographical areas.

## 7.3 Airport reference matching

The airport reference file does not provide an unambiguous coordinate match for every code. The project avoids guessing where a code is ambiguous, but this means some route pairs cannot be shown on the map. An improved airport reference dataset or a documented manual review of high-frequency unmatched codes could improve coverage.

## 7.4 Observation times and punctuality

The difference between `first_seen` and `last_seen` is an observation span. It is not confirmed airborne time, block time or a measure of delay. A punctuality study would require scheduled and actual operational timestamps, with clearly defined events such as scheduled departure, actual off-block, take-off, landing and in-block.

## 7.5 Further research opportunities

Several extensions would strengthen the project:

1. **Investigate missing endpoint patterns.** Determine whether incomplete ADEP and ADES information is concentrated in specific groups.
2. **Calculate record-weighted map coverage.** Measure what share of endpoint-complete records belongs to route pairs that can be mapped, rather than reporting only the share of distinct route pairs.
3. **Compare months and seasons.** Explore whether the monthly pattern differs across operators, airports or aircraft categories.
4. **Study network structure.** Calculate measures such as airport degree, route concentration and network centrality, while being explicit about the endpoint-complete subset.
5. **Add operational schedule data.** Combine flight-list records with an appropriate schedule or operational dataset if the aim is to study delay or punctuality.
6. **Improve aircraft categorisation.** Group typecodes into broader categories such as narrow-body, wide-body, regional aircraft and other types, using a documented classification source.


# 8. Conclusion

This project set out to describe observed flight-list activity during 2025 and to make the resulting analysis accessible through an interactive dashboard. It combines twelve monthly OPDI Flight List files, applies consistent data preparation and quality checks, and organises the accepted records into temporal, operator, airport, aircraft and route summaries.

The baseline processing retained 16,264,265 unique records. It also revealed an important limitation: only 34.3% of accepted records contained both departure and destination airport codes. Consequently, the route network and route rankings describe a smaller endpoint-complete subset of the data. Airport matching further determines which of those directional pairs can be represented geographically.

The project therefore makes two contributions. First, it provides a reproducible way to prepare and explore a large flight-list dataset. Second, it demonstrates why data completeness and careful indicator definitions are essential when drawing conclusions from surveillance-derived aviation data.

The Streamlit dashboard complements the notebook by allowing the prepared results to be explored interactively. Together, the report and dashboard provide a practical descriptive view of the flight activity represented in the 2025 OPDI Flight List, while making clear what the data can and cannot establish.


# 9. Streamlit dashboard

The Streamlit application is the interactive companion to this research report. It reads the prepared datasets produced by the notebook and presents the analysis through filters, charts, tables and a route-network map.

| Dashboard section | Research purpose |
|---|---|
| Overview | Explore total and monthly record activity |
| Network map | Explore the mapped directional airport-pair network |
| Airlines | Compare records by operator |
| Airports | Compare departure and arrival activity |
| Fleet | Explore aircraft type representation |
| Routes | Rank and filter directional airport pairs |
| Flight explorer | Inspect individual flight-list records |
| Data quality | Review missing fields and preparation checks |

The dashboard is designed to support exploration of the research questions. For example, a user can filter the route view to one operator or period, or compare departure and arrival counts for an airport.

The same definitions and caveats apply in the dashboard as in the report. Route maps show airport-pair connections, not actual aircraft trajectories. Route rankings use records with both endpoints. Operator and aircraft rankings use records with the relevant field available. The dashboard does not calculate punctuality or delay.


# 10. Data preparation and analysis code

The remaining sections contain the executable code used to prepare the data and generate updated results. The code is kept separate from the research narrative so that the research question, decisions and findings can be read without having to work through every implementation detail.

Run the code cells in order, from top to bottom, from the main project folder. The notebook expects the twelve monthly Parquet files and airport reference file in the folders described in Section 12. The original monthly files are read but not changed. Prepared files are written to `processed_data/`.

The code uses pandas for standard table operations: reading files, filtering records, grouping values and joining reference information. The preparation avoids repeating the same airport lookup for every flight record by first creating a list of distinct airport codes.


## 10.1 Project folders and input files

This section defines where the notebook expects to find the source files and where it should save the results. The output folder is created if it does not already exist. The downloaded monthly source files are not overwritten.


In [ ]:

from pathlib import Path
import json
import re
from datetime import datetime, timezone

import numpy as np
import pandas as pd

# Project folders. Keep raw monthly files unchanged.
PROJECT_DIR = Path.cwd()
FLIGHT_DIR = PROJECT_DIR / "data" / "flight_list"
AIRPORTS_FILE = PROJECT_DIR / "data" / "airports.csv"
OUTPUT_DIR = PROJECT_DIR / "processed_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# This notebook is configured for the complete 2025 study year.
EXPECTED_YEAR = 2025
EXPECTED_MONTHS = set(range(1, 13))

print("Project:", PROJECT_DIR.resolve())
print("Flight inputs:", FLIGHT_DIR.resolve())
print("Airport reference:", AIRPORTS_FILE.resolve())
print("Outputs:", OUTPUT_DIR.resolve())


In [ ]:

if not FLIGHT_DIR.exists():
    raise FileNotFoundError(f"Flight input folder not found: {FLIGHT_DIR}")

source_files = sorted(p for p in FLIGHT_DIR.glob("*.parquet") if p.is_file())
if not source_files:
    raise FileNotFoundError(f"No monthly Parquet files found in {FLIGHT_DIR}")

def month_from_filename(path):
    match = re.search(r"(20\d{2})(0[1-9]|1[0-2])", path.stem)
    if not match:
        return None
    return (int(match.group(1)), int(match.group(2)))

file_months = [month_from_filename(p) for p in source_files]
if any(x is None for x in file_months):
    bad = [p.name for p, m in zip(source_files, file_months) if m is None]
    raise ValueError(f"Could not identify YYYYMM in these filenames: {bad}")

observed_months = {m for year, m in file_months if year == EXPECTED_YEAR}
if len(source_files) != 12 or observed_months != EXPECTED_MONTHS or any(
    year != EXPECTED_YEAR for year, _ in file_months
):
    raise ValueError(
        f"Expected exactly 12 monthly files for {EXPECTED_YEAR}; "
        f"found {len(source_files)} files and months {sorted(observed_months)}."
    )

print(f"Input inventory confirmed: {len(source_files)} monthly files for {EXPECTED_YEAR}.")
for p in source_files:
    print(f"  {p.name}  ({p.stat().st_size / 1024**2:.1f} MB)")


## 10.2 Airport reference data

The airport reference file connects airport codes to names, countries and coordinates. A code that points to multiple different locations is treated as ambiguous rather than assigned a guessed coordinate. This is a deliberate accuracy choice: leaving a point unmapped is preferable to showing a route in the wrong location.


In [ ]:
if not AIRPORTS_FILE.exists():
    raise FileNotFoundError(
        f"Airport reference file not found: {AIRPORTS_FILE}. "
        "Place airports.csv in the project's data folder."
    )

airports_raw = pd.read_csv(AIRPORTS_FILE, low_memory=False)
required_airport_cols = {"latitude_deg", "longitude_deg"}
missing = required_airport_cols - set(airports_raw.columns)
if missing:
    raise ValueError(f"Airport reference is missing required columns: {sorted(missing)}")

for col in ["icao_code", "iata_code", "ident", "name", "type", "municipality", "iso_country"]:
    if col not in airports_raw.columns:
        airports_raw[col] = pd.NA
    airports_raw[col] = airports_raw[col].astype("string").str.strip()
    airports_raw[col] = airports_raw[col].replace("", pd.NA)

for col in ["icao_code", "iata_code", "ident"]:
    airports_raw[col] = airports_raw[col].str.upper()

airports_raw["latitude_deg"] = pd.to_numeric(airports_raw["latitude_deg"], errors="coerce")
airports_raw["longitude_deg"] = pd.to_numeric(airports_raw["longitude_deg"], errors="coerce")

valid_coord = (
    airports_raw["latitude_deg"].between(-90, 90)
    & airports_raw["longitude_deg"].between(-180, 180)
)
airports_ref = airports_raw.loc[valid_coord].copy()

# Restrict the map lookup to airports with a usable ICAO or IATA identifier.
airports_ref["icao_code"] = airports_ref["icao_code"].replace("", pd.NA)
airports_ref["iata_code"] = airports_ref["iata_code"].replace("", pd.NA)

# Build a deterministic ICAO lookup. If a code has multiple rows, keep it only
# when all candidate coordinates are identical; otherwise mark it ambiguous.
def build_code_lookup(frame, code_col):
    records = []
    for code, group in frame.dropna(subset=[code_col]).groupby(code_col):
        coord_pairs = group[["latitude_deg", "longitude_deg"]].drop_duplicates()
        if len(coord_pairs) == 1:
            row = group.sort_values(["type", "name"], na_position="last").iloc[0]
            records.append({
                "code": code,
                "latitude_deg": float(row["latitude_deg"]),
                "longitude_deg": float(row["longitude_deg"]),
                "airport_name": row["name"],
                "municipality": row["municipality"],
                "iso_country": row["iso_country"],
                "reference_ident": row["ident"],
                "candidate_rows": len(group),
                "ambiguous": False,
            })
        else:
            records.append({
                "code": code, "latitude_deg": np.nan, "longitude_deg": np.nan,
                "airport_name": pd.NA, "municipality": pd.NA,
                "iso_country": pd.NA, "reference_ident": pd.NA,
                "candidate_rows": len(group), "ambiguous": True,
            })
    return pd.DataFrame(records)

icao_lookup = build_code_lookup(airports_ref, "icao_code")
iata_lookup = build_code_lookup(airports_ref, "iata_code")

# For IATA, require a single source row for the code, not just coincident coordinates.
iata_counts = airports_ref.dropna(subset=["iata_code"]).groupby("iata_code").size()
iata_lookup["ambiguous"] = iata_lookup["code"].map(iata_counts).fillna(0).gt(1)

print("Airport reference rows:", len(airports_raw))
print("Rows with valid coordinates:", len(airports_ref))
print("ICAO lookup codes:", len(icao_lookup))
print("IATA lookup codes:", len(iata_lookup))
print("Ambiguous ICAO codes:", int(icao_lookup["ambiguous"].sum()))
print("Ambiguous IATA codes:", int(iata_lookup["ambiguous"].sum()))


## 10.3 Reading and standardising the monthly records

The monthly files are expected to have the same general structure. Column names are standardised, text codes are stripped of extra spaces and converted to uppercase, and dates and timestamps are converted to consistent datetime values. These steps prevent small formatting differences from creating separate categories.

The files are then combined into one table. The date of flight is taken from the source date field where possible; the first observation time is used only as a fallback when the date is missing.


In [ ]:

CODE_COLUMNS = [
    "icao_operator", "icao24", "adep", "ades", "typecode",
    "icao_aircraft_class", "registration", "model", "callsign", "flt_id"
]

def normalise_columns(frame):
    # The source files share the same schema; this handles case/spacing differences.
    frame = frame.copy()
    frame.columns = [str(c).strip().lower() for c in frame.columns]
    return frame

def normalise_id(series):
    out = series.astype("string").str.strip()
    return out.replace("", pd.NA)

def parse_mixed_timestamp(series):
    if pd.api.types.is_numeric_dtype(series):
        numeric = pd.to_numeric(series, errors="coerce")
        finite = numeric.dropna()
        if finite.empty:
            return pd.to_datetime(series, errors="coerce", utc=True)
        unit = "ms" if finite.abs().median() > 1e11 else "s"
        return pd.to_datetime(numeric, unit=unit, errors="coerce", utc=True)
    return pd.to_datetime(series, errors="coerce", utc=True)

def parse_dof(series):
    if pd.api.types.is_numeric_dtype(series):
        text = pd.to_numeric(series, errors="coerce").round().astype("Int64").astype("string")
        return pd.to_datetime(text, format="%Y%m%d", errors="coerce", utc=True)
    text = series.astype("string").str.strip()
    compact = pd.to_datetime(text, format="%Y%m%d", errors="coerce", utc=True)
    general = pd.to_datetime(text, errors="coerce", utc=True)
    return compact.fillna(general)


In [ ]:

frames = []

for path in source_files:
    raw = pd.read_parquet(path)
    frame = normalise_columns(raw)

    required = {"id", "adep", "ades", "icao_operator"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{path.name} is missing required fields: {sorted(missing)}")

    frame["id"] = normalise_id(frame["id"])
    frame = frame.dropna(subset=["id"]).copy()

    for col in CODE_COLUMNS:
        if col in frame.columns:
            frame[col] = frame[col].astype("string").str.strip().replace("", pd.NA)
            if col in [
                "icao_operator", "icao24", "adep", "ades",
                "typecode", "icao_aircraft_class"
            ]:
                frame[col] = frame[col].str.upper()

    frame["dof_dt"] = parse_dof(frame["dof"]) if "dof" in frame else pd.NaT
    for col in ["first_seen", "last_seen"]:
        frame[col + "_dt"] = (
            parse_mixed_timestamp(frame[col]) if col in frame else pd.NaT
        )

    # Prefer OPDI's date-of-flight field; use first_seen only as a fallback.
    frame["flight_datetime_utc"] = frame["dof_dt"]
    fallback = frame["flight_datetime_utc"].isna() & frame["first_seen_dt"].notna()
    frame.loc[fallback, "flight_datetime_utc"] = frame.loc[fallback, "first_seen_dt"]
    frame["flight_date"] = frame["flight_datetime_utc"].dt.date
    frame["flight_month"] = frame["flight_datetime_utc"].dt.strftime("%Y-%m")

    # This is an observation span, not a confirmed gate-to-gate or airborne duration.
    frame["observation_span_min"] = (
        (frame["last_seen_dt"] - frame["first_seen_dt"]).dt.total_seconds() / 60
    )
    frame.loc[frame["observation_span_min"] <= 0, "observation_span_min"] = np.nan

    frame["source_file"] = path.name
    frames.append(frame)

all_flights = pd.concat(frames, ignore_index=True, sort=False)
print(f"Rows loaded after removing missing IDs: {len(all_flights):,}")
print(f"Distinct flight IDs before duplicate handling: {all_flights['id'].nunique():,}")
print(f"Rows with both ADEP and ADES before duplicate handling: "
      f"{(all_flights['adep'].notna() & all_flights['ades'].notna()).sum():,}")


## 10.4 Duplicate and conflicting-ID checks

Exact repeated rows are removed before the analysis. If an ID still appears with different information, it is treated as conflicting and excluded from the accepted table. The excluded IDs are retained in an exception file so that the decision is visible and can be reviewed.


In [ ]:

# Remove exact repeated rows first. Then quarantine any ID that still has
# non-identical records; do not choose one version arbitrarily.
before_dedup = len(all_flights)
comparison_columns = [c for c in all_flights.columns if c != "source_file"]
dedup_exact = all_flights.drop_duplicates(subset=comparison_columns).copy()
exact_duplicates_removed = before_dedup - len(dedup_exact)

duplicate_rows = dedup_exact.loc[dedup_exact["id"].duplicated(keep=False)].copy()
conflict_rows = []
conflict_ids = []

for flight_id, group in duplicate_rows.groupby("id", sort=False):
    differing = []
    for col in dedup_exact.columns:
        if col in {"id", "source_file"}:
            continue
        values = group[col].astype("string").fillna("<NULL>").unique()
        if len(values) > 1:
            differing.append(col)
    if differing:
        conflict_ids.append(flight_id)
        conflict_rows.append({
            "id": str(flight_id),
            "source_files": "; ".join(sorted(group["source_file"].dropna().unique())),
            "conflicting_fields": "; ".join(differing),
        })

duplicate_conflicts = pd.DataFrame(
    conflict_rows, columns=["id", "source_files", "conflicting_fields"]
)
conflict_id_set = set(conflict_ids)
flights = dedup_exact.loc[~dedup_exact["id"].isin(conflict_id_set)].copy()

assert flights["id"].is_unique, "Accepted flight IDs are not unique."
print(f"Exact duplicate rows removed: {exact_duplicates_removed:,}")
print(f"Conflicting IDs excluded: {len(conflict_id_set):,}")
print(f"Accepted unique flight records: {len(flights):,}")


## 10.5 Matching airport codes

Airport matching is performed for the distinct codes found in ADEP and ADES. The matched airport name and coordinates are then added to the flight records. Missing and ambiguous codes remain visible in the data-quality output; they are not silently replaced with guessed locations.


In [ ]:
icao_map = icao_lookup.set_index("code").to_dict("index")
iata_map = iata_lookup.set_index("code").to_dict("index")

def lookup_airport(code):
    if pd.isna(code):
        return {
            "latitude_deg": np.nan, "longitude_deg": np.nan,
            "airport_name": pd.NA, "municipality": pd.NA,
            "iso_country": pd.NA, "match_method": "missing_code",
            "match_status": "unmatched",
        }
    code = str(code).strip().upper()
    if len(code) == 4 and code in icao_map:
        item = icao_map[code]
        if not item["ambiguous"]:
            return {**item, "match_method": "icao_code", "match_status": "matched"}
        return {
            "latitude_deg": np.nan, "longitude_deg": np.nan,
            "airport_name": pd.NA, "municipality": pd.NA, "iso_country": pd.NA,
            "match_method": "icao_code", "match_status": "ambiguous",
        }
    if len(code) == 3 and code in iata_map:
        item = iata_map[code]
        if not item["ambiguous"]:
            return {**item, "match_method": "iata_code", "match_status": "matched"}
        return {
            "latitude_deg": np.nan, "longitude_deg": np.nan,
            "airport_name": pd.NA, "municipality": pd.NA, "iso_country": pd.NA,
            "match_method": "iata_code", "match_status": "ambiguous",
        }
    return {
        "latitude_deg": np.nan, "longitude_deg": np.nan,
        "airport_name": pd.NA, "municipality": pd.NA, "iso_country": pd.NA,
        "match_method": "no_reference_match", "match_status": "unmatched",
    }

# Unique code dimension avoids repeated lookups for every flight.
all_codes = pd.Series(pd.concat([flights["adep"], flights["ades"]], ignore_index=True).dropna().unique())
airport_match_rows = []
for code in all_codes:
    airport_match_rows.append({"airport_code": code, **lookup_airport(code)})
airport_match = pd.DataFrame(airport_match_rows)

# Attach origin and destination metadata independently.
airport_meta = airport_match.set_index("airport_code")
for role, code_col in [("origin", "adep"), ("destination", "ades")]:
    flights[f"{role}_latitude"] = flights[code_col].map(airport_meta["latitude_deg"])
    flights[f"{role}_longitude"] = flights[code_col].map(airport_meta["longitude_deg"])
    flights[f"{role}_name"] = flights[code_col].map(airport_meta["airport_name"])
    flights[f"{role}_match_status"] = flights[code_col].map(airport_meta["match_status"])
    flights[f"{role}_match_method"] = flights[code_col].map(airport_meta["match_method"])

# Build one row per distinct code that could not be mapped.
unmatched_codes = airport_match.loc[
    airport_match["match_status"] != "matched"
].copy().sort_values("airport_code")

print("Unique airport codes used by flights:", len(airport_match))
print("Matched codes:", int((airport_match["match_status"] == "matched").sum()))
print("Unmatched or ambiguous codes:", len(unmatched_codes))
display(unmatched_codes.head(30))


## 10.6 Creating directional route summaries

A route summary is created only for records with both ADEP and ADES. The ordered pair is retained, so the two directions between the same airports remain separate. The summary includes the number of flight-list records associated with each pair and whether both airports have usable coordinates.


In [ ]:

route_source = flights.dropna(subset=["adep", "ades"]).copy()
route_source = route_source.loc[
    route_source["adep"].ne("") & route_source["ades"].ne("")
].copy()

route_agg = {
    "flight_count": ("id", "nunique"),
    "median_observation_span_min": ("observation_span_min", "median"),
}
for col, output in [
    ("icao24", "unique_aircraft"),
    ("icao_operator", "unique_operators"),
    ("typecode", "unique_typecodes"),
]:
    if col in route_source.columns:
        route_agg[output] = (col, "nunique")

route_pairs = (
    route_source.groupby(["adep", "ades"])
    .agg(**route_agg)
    .reset_index()
)

origin_dim = airport_match.rename(columns={
    "airport_code": "adep",
    "latitude_deg": "origin_latitude",
    "longitude_deg": "origin_longitude",
    "airport_name": "origin_airport_name",
    "municipality": "origin_municipality",
    "iso_country": "origin_country",
    "match_status": "origin_match_status",
    "match_method": "origin_match_method",
})[[
    "adep", "origin_latitude", "origin_longitude", "origin_airport_name",
    "origin_municipality", "origin_country", "origin_match_status",
    "origin_match_method"
]]

destination_dim = airport_match.rename(columns={
    "airport_code": "ades",
    "latitude_deg": "destination_latitude",
    "longitude_deg": "destination_longitude",
    "airport_name": "destination_airport_name",
    "municipality": "destination_municipality",
    "iso_country": "destination_country",
    "match_status": "destination_match_status",
    "match_method": "destination_match_method",
})[[
    "ades", "destination_latitude", "destination_longitude",
    "destination_airport_name", "destination_municipality",
    "destination_country", "destination_match_status",
    "destination_match_method"
]]

route_pairs = route_pairs.merge(
    origin_dim, on="adep", how="left", validate="many_to_one"
).merge(
    destination_dim, on="ades", how="left", validate="many_to_one"
)

route_pairs["route"] = route_pairs["adep"].astype(str) + " → " + route_pairs["ades"].astype(str)
route_pairs["route_key"] = route_pairs["adep"].astype(str) + "|" + route_pairs["ades"].astype(str)
route_pairs["is_mapped"] = (
    route_pairs["origin_latitude"].notna()
    & route_pairs["origin_longitude"].notna()
    & route_pairs["destination_latitude"].notna()
    & route_pairs["destination_longitude"].notna()
)
route_pairs = route_pairs.sort_values(
    ["flight_count", "adep", "ades"], ascending=[False, True, True]
).reset_index(drop=True)

print(f"Flights with both route endpoints: {len(route_source):,}")
print(f"Unique directed routes: {len(route_pairs):,}")
print(f"Mapped directed routes: {int(route_pairs['is_mapped'].sum()):,}")


## 10.7 Creating airport and monthly summaries

Departure and arrival counts are calculated separately. A record contributes to a departure count when ADEP is known and to an arrival count when ADES is known. Monthly summaries count records by month and provide additional descriptive counts where the relevant fields are available.


In [ ]:

# Airport activity: each known ADEP contributes one departure; each known ADES
# contributes one arrival. These are flight-list record counts, not passenger counts.
flights["route_key"] = (
    flights["adep"].astype("string").fillna("<MISSING>")
    + "|" + flights["ades"].astype("string").fillna("<MISSING>")
)

departures = (
    flights.dropna(subset=["adep"]).groupby("adep")["id"].nunique()
    .rename("departures")
)
arrivals = (
    flights.dropna(subset=["ades"]).groupby("ades")["id"].nunique()
    .rename("arrivals")
)

airport_summary = pd.concat([departures, arrivals], axis=1).fillna(0).reset_index()
airport_summary = airport_summary.rename(columns={airport_summary.columns[0]: "airport_code"})
airport_summary["departures"] = airport_summary["departures"].astype("int64")
airport_summary["arrivals"] = airport_summary["arrivals"].astype("int64")
airport_summary["total_movements"] = (
    airport_summary["departures"] + airport_summary["arrivals"]
)
airport_summary = airport_summary.merge(
    airport_match[[
        "airport_code", "latitude_deg", "longitude_deg",
        "airport_name", "municipality", "iso_country", "match_status"
    ]],
    on="airport_code", how="left", validate="one_to_one"
).sort_values("total_movements", ascending=False).reset_index(drop=True)

dated = flights.dropna(subset=["flight_month"]).copy()
monthly_summary = (
    dated.groupby("flight_month")
    .agg(
        flights=("id", "nunique"),
        directed_route_pairs=("route_key", "nunique"),
        operators=("icao_operator", "nunique"),
        aircraft=("icao24", "nunique"),
    )
    .reset_index()
    .sort_values("flight_month")
)

display(monthly_summary)
display(airport_summary.head(10))
display(route_pairs.head(10))


## 10.8 Validation and audit files

The notebook checks that accepted IDs and route pairs are unique, route counts reconcile with the endpoint-complete records, coordinates are valid, and airport departure and arrival totals reconcile with the source records. These checks test internal consistency; they do not establish that the source captures every flight.


In [ ]:

# Keep a short set of checks that protect the outputs' integrity.
# The earlier full pipeline already established the processing behaviour;
# these checks remain as safeguards if the source files are refreshed.
validation_results = []

def check(name, passed, detail):
    validation_results.append({
        "check": name,
        "passed": bool(passed),
        "detail": str(detail),
    })
    if not passed:
        raise AssertionError(f"Validation failed — {name}: {detail}")

check(
    "Accepted flight IDs are unique",
    flights["id"].is_unique,
    f"{flights['id'].nunique():,} unique IDs / {len(flights):,} rows"
)
check(
    "Directed route keys are unique",
    route_pairs["route_key"].is_unique,
    f"{route_pairs['route_key'].nunique():,} unique route rows"
)
check(
    "Route counts reconcile",
    int(route_pairs["flight_count"].sum()) == len(route_source),
    f"{int(route_pairs['flight_count'].sum()):,} route-linked records / "
    f"{len(route_source):,} eligible records"
)
mapped = route_pairs.loc[route_pairs["is_mapped"]]
coordinates_valid = (
    mapped["origin_latitude"].between(-90, 90).all()
    and mapped["destination_latitude"].between(-90, 90).all()
    and mapped["origin_longitude"].between(-180, 180).all()
    and mapped["destination_longitude"].between(-180, 180).all()
)
check(
    "Mapped airport coordinates are within valid ranges",
    coordinates_valid,
    f"{len(mapped):,} mapped directed route pairs checked"
)
check(
    "Airport movement totals reconcile",
    int(airport_summary["departures"].sum()) == int(flights["adep"].notna().sum())
    and int(airport_summary["arrivals"].sum()) == int(flights["ades"].notna().sum()),
    "Departure and arrival summaries match records with known endpoints"
)

validation_df = pd.DataFrame(validation_results)
display(validation_df)


In [ ]:

# Build the unmatched-airport audit with the number of affected records.
unmatched_origin = (
    flights.loc[flights["origin_match_status"].ne("matched"), "adep"]
    .value_counts().rename_axis("airport_code").reset_index(name="origin_flight_count")
)
unmatched_destination = (
    flights.loc[flights["destination_match_status"].ne("matched"), "ades"]
    .value_counts().rename_axis("airport_code").reset_index(name="destination_flight_count")
)
unmatched_airports = unmatched_codes.merge(
    unmatched_origin, on="airport_code", how="left"
).merge(
    unmatched_destination, on="airport_code", how="left"
)
for col in ["origin_flight_count", "destination_flight_count"]:
    unmatched_airports[col] = unmatched_airports[col].fillna(0).astype("int64")

date_min = flights["flight_datetime_utc"].min()
date_max = flights["flight_datetime_utc"].max()
routeable_count = len(route_source)
mapped_route_count = int(route_pairs["is_mapped"].sum())

report = {
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "study_year": EXPECTED_YEAR,
    "source_files": [p.name for p in source_files],
    "source_file_count": len(source_files),
    "source_rows_after_missing_id_removal": int(len(all_flights)),
    "exact_duplicate_rows_removed": int(exact_duplicates_removed),
    "conflicting_flight_ids_excluded": int(len(conflict_id_set)),
    "accepted_flight_rows": int(len(flights)),
    "accepted_unique_flight_ids": int(flights["id"].nunique()),
    "flights_with_both_route_endpoints": int(routeable_count),
    "route_endpoint_coverage_pct": round(100 * routeable_count / len(flights), 2),
    "unique_directed_route_pairs": int(len(route_pairs)),
    "mapped_directed_route_pairs": mapped_route_count,
    "unmapped_directed_route_pairs": int(len(route_pairs) - mapped_route_count),
    "mapped_route_pair_share_pct": round(100 * mapped_route_count / len(route_pairs), 2)
        if len(route_pairs) else None,
    "unique_airport_codes_in_flights": int(len(airport_match)),
    "unmatched_or_ambiguous_airport_codes": int(len(unmatched_codes)),
    "flights_without_flight_date": int(flights["flight_datetime_utc"].isna().sum()),
    "flight_date_min_utc": None if pd.isna(date_min) else date_min.isoformat(),
    "flight_date_max_utc": None if pd.isna(date_max) else date_max.isoformat(),
    "validation_checks": validation_results,
    "notes": [
        "The flight-list records are OPDI trajectory records, not passenger counts.",
        "Route pairs are directional ADEP-to-ADES combinations.",
        "Airport coordinates are airport reference points, not aircraft positions.",
        "Map links show airport-pair relationships, not flown trajectories.",
        "last_seen minus first_seen is an observation span, not confirmed block time.",
    ],
}

print(json.dumps(report, indent=2))


## 10.9 Saving the prepared data

The notebook saves the prepared flight-level records and the smaller summary tables used by the dashboard. It also saves exception files, validation results and a compact JSON run report. Parquet is used for the larger tables because it is suitable for efficient storage and reading; CSV is used for small audit tables because they are easy to inspect.


In [ ]:

# Keep only fields used by the dashboard and the core analysis.
# This avoids carrying redundant intermediate timestamps, route keys and
# unused source columns into the 16-million-row flight-level export.
flight_output_columns = [
    "id", "flight_date", "icao_operator", "icao24", "flt_id", "callsign",
    "adep", "ades", "registration", "model", "typecode", "icao_aircraft_class",
    "observation_span_min",
    "origin_latitude", "origin_longitude", "origin_name", "origin_match_status",
    "destination_latitude", "destination_longitude", "destination_name",
    "destination_match_status",
]
flights["origin_name"] = flights.get("origin_name", flights.get("origin_airport_name", pd.NA))
flights["destination_name"] = flights.get(
    "destination_name", flights.get("destination_airport_name", pd.NA)
)
selected_columns = [c for c in flight_output_columns if c in flights.columns]
flights_export = flights[selected_columns].copy()
flights_export["flight_date"] = pd.to_datetime(
    flights_export["flight_date"], errors="coerce"
)

outputs = {
    "flights_processed.parquet": flights_export,
    "route_pairs.parquet": route_pairs,
    "airport_summary.parquet": airport_summary,
    "monthly_summary.parquet": monthly_summary,
}

# Convert mixed Python object columns to consistent nullable strings for Parquet.
for filename, frame in outputs.items():
    for col in frame.select_dtypes(include=["object"]).columns:
        frame[col] = frame[col].astype("string")
    target = OUTPUT_DIR / filename
    frame.to_parquet(target, index=False, compression="snappy")
    print(f"{target.name}: {len(frame):,} rows, {target.stat().st_size / 1024**2:.1f} MB")

# Keep only compact exception and validation outputs used for review/dashboard QA.
duplicate_conflicts.to_csv(OUTPUT_DIR / "duplicate_conflicts.csv", index=False)
unmatched_airports.to_csv(OUTPUT_DIR / "unmatched_airports.csv", index=False)
validation_df.to_csv(OUTPUT_DIR / "validation_results.csv", index=False)
(OUTPUT_DIR / "preprocessing_report.json").write_text(
    json.dumps(report, indent=2, default=str), encoding="utf-8"
)

print("Processing complete:", OUTPUT_DIR.resolve())


# 11. Results generated from the current run

The following tables and charts are generated from the data prepared when the code above is executed. They should be compared with the baseline figures in Section 5. If the results differ, investigate the difference before replacing any processed data currently used by the dashboard.

The baseline values in Section 5 are historical reference values. The results below reflect the current execution of this notebook.


In [ ]:

PLACEHOLDERS = {"", "NONE", "NULL", "NAN", "UNKNOWN", "N/A", "NA"}

def clean_category(series):
    values = series.astype("string").str.strip()
    return values[values.notna() & ~values.str.upper().isin(PLACEHOLDERS)]

def top_counts(series, label, count_label="flight_records", n=10):
    clean = clean_category(series)
    result = clean.value_counts().head(n).rename_axis(label).reset_index(name=count_label)
    result["share_of_known_records_pct"] = (
        100 * result[count_label] / clean.shape[0]
    ).round(2)
    return result

# population
quality_rows = []
for col in ["icao_operator", "adep", "ades", "icao24", "typecode", "model", "registration"]:
    if col in flights.columns:
        known = int(flights[col].notna().sum())
        quality_rows.append({
            "field": col,
            "records_with_value": known,
            "records_missing": int(len(flights) - known),
            "completeness_pct": round(100 * known / len(flights), 2),
        })
field_completeness = pd.DataFrame(quality_rows).sort_values("completeness_pct", ascending=True)

# rankings
top_operators = top_counts(flights["icao_operator"], "operator_code")
top_aircraft_types = top_counts(
    flights.loc[
        ~flights.get("typecode", pd.Series(index=flights.index, dtype="string"))
        .astype("string").str.upper().eq("GRND")
        & ~flights.get("model", pd.Series(index=flights.index, dtype="string"))
        .astype("string").str.upper().str.contains("GROUND VEHICLE", na=False),
        "typecode"
    ],
    "typecode"
)

top_departure_airports = top_counts(flights["adep"], "departure_airport")
top_arrival_airports = top_counts(flights["ades"], "arrival_airport")

top_routes = route_pairs[
    ["route", "adep", "ades", "flight_count", "is_mapped"]
].head(10).copy()

monthly_results = monthly_summary.copy()
monthly_results["flight_month"] = monthly_results["flight_month"].astype(str)

print(f"Accepted unique flight-list records: {len(flights):,}")
print(f"Records with both airport endpoints: {len(route_source):,} "
      f"({100 * len(route_source) / len(flights):.1f}%)")
print(f"Distinct directed airport pairs: {len(route_pairs):,}")
print(f"Mapped directed airport pairs: {int(route_pairs['is_mapped'].sum()):,} "
      f"({100 * route_pairs['is_mapped'].mean():.1f}%)")

display(field_completeness)
display(monthly_results)
display(top_operators)
display(top_departure_airports)
display(top_arrival_airports)
display(top_aircraft_types)
display(top_routes)

In [ ]:
# compact figures
import matplotlib.pyplot as plt

monthly_plot = monthly_results.copy()
monthly_plot["month"] = pd.to_datetime(monthly_plot["flight_month"] + "-01")
monthly_plot = monthly_plot.sort_values("month")


fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(monthly_plot["month"], monthly_plot["flights"], marker="o")
ax.set_title("Observed OPDI flight list by month : 2025")
ax.set_xlabel("Month")
ax.set_ylabel("Flight-list records")
ax.grid(True, alpha=0.25)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()



fig, ax = plt.subplots(figsize=(10, 5))
plot_ops = top_operators.sort_values("flight_records")
ax.barh(plot_ops["operator_code"], plot_ops["flight_records"])
ax.set_title("Top operator codes by accepted flight-list records")
ax.set_xlabel("Flight-list records")
ax.set_ylabel("Operator code")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

# 12. Running the notebook

See README.md

# 13. References

1. Open Performance Data Initiative (OPDI). *Data catalogue*. https://www.opdi.aero/data
2. Open Performance Data Initiative (OPDI). *Flight List data*. https://www.opdi.aero/flight-list-data
3. Open Performance Data Initiative (OPDI). *Concepts*. https://www.opdi.aero/concepts
4. Open Performance Data Initiative (OPDI). *Methodology*. https://www.opdi.aero/methodology
5. EUROCONTROL Performance Review Commission and OpenSky Network. *The Open Performance Data Initiative: A Foundation Supporting the European Open Science Alliance for ATM Research*. SESAR Innovation Days, 2024. https://sesar.eu/sites/default/files/documents/sid/2024/papers/SIDs_2024_paper_100%20final.pdf
6. Olive, X., Krummen, J., Figuet, B., & Alligier, R. (2025). Filtering Techniques for ADS-B Trajectory Preprocessing. *Journal of Open Aviation Science*, 2(2). https://doi.org/10.59490/joas.2024.7882
7. Apache Software Foundation. *Apache Parquet Documentation*. https://parquet.apache.org/
8. International Civil Aviation Organization (ICAO). *Location Indicators, Doc 7910*. https://www.icao.int/operational-safety/Designators-and-indicators




# 14. AI Statement

AI was used during this project in a supportive way. Initially, the dataset was structured in a .paruqet format, for which I hadno previous experience with. Hence I used AI do read the files and convert them into managable formats. In addition, in terms of coding, AI has helped with testing the methods used to process the data. While working with 12 datasets, there occurred multiple problems when trying to merge and trying to calculate correctly flight list pairs. However, flight pairs, outputs and observed patters have been revised manually through multiple iterations to ensure that they resemble a realistic picture of the network. In terms of streamlit, although it was introduced in the course, I used AI to add interesting visuals and more advanced visualization techniques which I think did enhance the look of the dashboard. 

However, the originality, research, validation and framework is the original work of the author. Full responsibility is assumed for the analysis and its conclusions.